# Part 2 - Advanced Pandas

This notebook builds on Part 1's introduction to pandas, covering **merging datasets**, **pivot tables**, **multi-column grouping**, and **applying custom functions** - the tools you need for real multi-file experimental datasets.

In [ ]:
import pandas as pd

tests = pd.read_csv('tensile_test_data.csv')
properties = pd.read_csv('material_properties.csv')

print(tests)
print(properties)

## Merging DataFrames
Real data often lives in multiple files that share a common column (a **key**). `pd.merge()` joins two DataFrames together on that key, similar to a SQL `JOIN`.

In [ ]:
merged = pd.merge(tests, properties, on='Material', how='left')
print(merged)

The `how` parameter controls which rows are kept:
- `'inner'`: only rows with matching keys in both DataFrames.
- `'left'`: all rows from the left DataFrame, matched where possible.
- `'outer'`: all rows from both, with `NaN` where there's no match.

## Computing a Derived Column After Merging
Now that we have `YoungsModulus_GPa` alongside our test data, we can compute engineering quantities that combine both datasets, such as estimated strain from stress and modulus (Hooke's Law: $\sigma = E \varepsilon$).

In [ ]:
area_mm2 = 12.5
merged['Stress_MPa'] = merged['PeakLoad_N'] / area_mm2
merged['Estimated_Strain'] = merged['Stress_MPa'] / (merged['YoungsModulus_GPa'] * 1000)
print(merged[['SampleID', 'Material', 'Stress_MPa', 'Estimated_Strain']])

## Grouping by Multiple Columns and Multiple Aggregations
`.groupby()` accepts a list of columns, and `.agg()` lets you compute several statistics at once.

In [ ]:
summary = merged.groupby('Material').agg(
    mean_stress_mpa=('Stress_MPa', 'mean'),
    max_stress_mpa=('Stress_MPa', 'max'),
    num_samples=('SampleID', 'count'),
)
print(summary)

## Pivot Tables
A **pivot table** reshapes data so that unique values from one column become new columns - useful for comparing groups side by side.

In [ ]:
pivot = merged.pivot_table(
    values='Stress_MPa',
    index='Material',
    aggfunc=['mean', 'max', 'min'],
)
print(pivot)

## Applying a Custom Function with `.apply()`
When a calculation is too complex for a single vectorized expression, `.apply()` lets you run a Python function on each row (or column).

In [ ]:
def classify_strength(row):
    """Labels a sample based on its stress relative to its material's typical range."""
    if row['Stress_MPa'] > 3000:
        return 'High'
    elif row['Stress_MPa'] > 2000:
        return 'Medium'
    else:
        return 'Low'

merged['Strength_Class'] = merged.apply(classify_strength, axis=1)
print(merged[['SampleID', 'Material', 'Stress_MPa', 'Strength_Class']])

## Saving Results
After processing, you'll often want to save your results for use in a report, a plot, or a downstream script (like one running on the SCC).

In [ ]:
merged.to_csv('merged_results.csv', index=False)
print("Saved merged_results.csv")

### *Exercise*
1. Merge `tests` and `properties` using `how='inner'` instead of `'left'` and note any difference.
2. Build a pivot table showing the **mean** `Estimated_Strain` for each `Material`.
3. Write your own function and use `.apply()` to flag any sample whose `Elongation_mm` exceeds 12.0 as `'High Ductility'`.

In [ ]:
# Enter your code here